In [10]:
import pandas as pd
import numpy as np
data = [10, 12, 12, 13, 13, 14, 12, 14, 13, 19, 90]
df=pd.DataFrame(data,columns=['value'])
df['z-score']=(df['value']-df['value'].mean())/df['value'].std()
df['z-score']
outlier_z=df[df['z-score'].abs()>3]
print(outlier_z)

    value   z-score
10     90  3.001264


In [12]:
Q1=df['value'].quantile(0.25)
Q3=df['value'].quantile(0.75)
IQR=Q3-Q1
IQR
df_iqr=df[(df['value'] < Q1-1.5*IQR) | (df['value']> Q3+1.5*IQR)]
df_iqr

,value,z-score
9,19,-0.050803
10,90,3.001264


In [18]:
from sklearn.preprocessing import PowerTransformer
df['log']=np.log(df['value'])
df['sqrt']=np.sqrt(df['value'])
pt_yeojohnson=PowerTransformer(method='yeo-johnson')
df['yeojohnson']=pt_yeojohnson.fit_transform(df[['value']])
df

,value,z-score,log,sqrt,yeojohnson
0,10,-0.437684,2.302585,3.162278,-1.878199
1,12,-0.351711,2.484907,3.464102,-0.557099
2,12,-0.351711,2.484907,3.464102,-0.557099
3,13,-0.308724,2.564949,3.605551,-0.115632
4,13,-0.308724,2.564949,3.605551,-0.115632
5,14,-0.265737,2.639057,3.741657,0.232733
6,12,-0.351711,2.484907,3.464102,-0.557099
7,14,-0.265737,2.639057,3.741657,0.232733
8,13,-0.308724,2.564949,3.605551,-0.115632
9,19,-0.050803,2.944439,4.358899,1.211874


Create synthetic Dataset with Missing value

In [5]:
import numpy as np
import pandas as pd

np.random.seed(0)

df=pd.DataFrame({
    "age":np.random.randint(18,70,size=10),
    "salary":np.random.randint(50000,100000,size=10),
    "city":['A','B','C','A','A','C','B','C','B','B'],
    "join_dates":pd.date_range("2020-02-01",periods=10)
})
df

,age,salary,city,join_dates
0,62,96884,A,2020-02-01
1,65,64935,B,2020-02-02
2,18,65430,C,2020-02-03
3,21,98600,A,2020-02-04
4,21,89512,A,2020-02-05
5,57,64650,C,2020-02-06
6,27,67089,B,2020-02-07
7,37,82230,C,2020-02-08
8,39,68983,B,2020-02-09
9,68,93095,B,2020-02-10


In [7]:
df.loc[[1,2,5],"age"]=np.nan
df.loc[[2,7],"salary"]=np.nan
df.loc[[3,5],"city"]=np.nan
df

,age,salary,city,join_dates
0,62.0,96884.0,A,2020-02-01
1,NaN,64935.0,B,2020-02-02
2,NaN,NaN,C,2020-02-03
3,21.0,98600.0,NaN,2020-02-04
4,21.0,89512.0,A,2020-02-05
5,NaN,64650.0,NaN,2020-02-06
6,27.0,67089.0,B,2020-02-07
7,37.0,NaN,C,2020-02-08
8,39.0,68983.0,B,2020-02-09
9,68.0,93095.0,B,2020-02-10


Inspect & quantify missingness

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   age         7 non-null      float64       
 1   salary      8 non-null      float64       
 2   city        8 non-null      str           
 3   join_dates  10 non-null     datetime64[us]
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 452.0 bytes


In [10]:
df.isnull().sum()

age           3
salary        2
city          2
join_dates    0
dtype: int64

In [11]:
df.isnull().mean()*100

age           30.0
salary        20.0
city          20.0
join_dates     0.0
dtype: float64

Strategy - Drop column and rows by threshold

In [14]:
th=0.40
drop_col=df.columns[df.isnull().mean()>th]
df.drop(columns=drop_col)
df

,age,salary,city,join_dates
0,62.0,96884.0,A,2020-02-01
1,NaN,64935.0,B,2020-02-02
2,NaN,NaN,C,2020-02-03
3,21.0,98600.0,NaN,2020-02-04
4,21.0,89512.0,A,2020-02-05
5,NaN,64650.0,NaN,2020-02-06
6,27.0,67089.0,B,2020-02-07
7,37.0,NaN,C,2020-02-08
8,39.0,68983.0,B,2020-02-09
9,68.0,93095.0,B,2020-02-10


In [29]:
df[df.isnull().sum(axis=1)<2]

,age,salary,city,join_dates
0,62.0,96884.0,A,2020-02-01
1,37.0,64935.0,B,2020-02-02
2,37.0,NaN,C,2020-02-03
3,21.0,98600.0,37.0,2020-02-04
4,21.0,89512.0,A,2020-02-05
5,37.0,64650.0,37.0,2020-02-06
6,27.0,67089.0,B,2020-02-07
7,37.0,NaN,C,2020-02-08
8,39.0,68983.0,B,2020-02-09
9,68.0,93095.0,B,2020-02-10


Simple imputation

In [27]:
df['age']=df['age'].fillna(df['age'].median())
df
df['city']=df['city'].fillna(df['city'].mode()[0])
df

,age,salary,city,join_dates
0,62.0,96884.0,A,2020-02-01
1,37.0,64935.0,B,2020-02-02
2,37.0,NaN,C,2020-02-03
3,21.0,98600.0,37.0,2020-02-04
4,21.0,89512.0,A,2020-02-05
5,37.0,64650.0,37.0,2020-02-06
6,27.0,67089.0,B,2020-02-07
7,37.0,NaN,C,2020-02-08
8,39.0,68983.0,B,2020-02-09
9,68.0,93095.0,B,2020-02-10


In [32]:
df.loc[[3],'city']='A'
df.loc[[5],'city']='C'
df

,age,salary,city,join_dates
0,62.0,96884.0,A,2020-02-01
1,37.0,64935.0,B,2020-02-02
2,37.0,NaN,C,2020-02-03
3,21.0,98600.0,A,2020-02-04
4,21.0,89512.0,A,2020-02-05
5,37.0,64650.0,C,2020-02-06
6,27.0,67089.0,B,2020-02-07
7,37.0,NaN,C,2020-02-08
8,39.0,68983.0,B,2020-02-09
9,68.0,93095.0,B,2020-02-10


In [39]:
df2=pd.DataFrame({
    "age":np.random.randint(18,70,size=10),
    "salary":np.random.randint(50000,100000,size=10),
    "city":['A','B','C','A','A','C','B','C','B','B'],
    "join_dates":pd.date_range("2020-02-01",periods=10)
})
df2.loc[[1,2,5],"age"]=np.nan
df2.loc[[2,7],"salary"]=np.nan
df2.loc[[3,5],"city"]=np.nan
df2

,age,salary,city,join_dates
0,39.0,91216.0,A,2020-02-01
1,NaN,86530.0,B,2020-02-02
2,NaN,NaN,C,2020-02-03
3,23.0,66221.0,NaN,2020-02-04
4,59.0,68819.0,A,2020-02-05
5,NaN,84402.0,NaN,2020-02-06
6,18.0,98682.0,B,2020-02-07
7,49.0,NaN,C,2020-02-08
8,23.0,70848.0,B,2020-02-09
9,48.0,60215.0,B,2020-02-10


In [42]:
from sklearn.impute import SimpleImputer

num_imp=SimpleImputer(strategy="median")
cat_imp_mfreq=SimpleImputer(strategy="most_frequent")
cat_imp_const=SimpleImputer(strategy="constant",fill_value="Missing")

df2['age']=num_imp.fit_transform(df2[['age']])[:,0]
df2['city']=cat_imp_mfreq.fit_transform(df2[['city']])[:,0]
df2

,age,salary,city,join_dates
0,39.0,91216.0,A,2020-02-01
1,39.0,86530.0,B,2020-02-02
2,39.0,NaN,C,2020-02-03
3,23.0,66221.0,B,2020-02-04
4,59.0,68819.0,A,2020-02-05
5,39.0,84402.0,B,2020-02-06
6,18.0,98682.0,B,2020-02-07
7,49.0,NaN,C,2020-02-08
8,23.0,70848.0,B,2020-02-09
9,48.0,60215.0,B,2020-02-10


In [48]:
df3=pd.DataFrame({
    "age":np.random.randint(18,70,size=10),
    "salary":np.random.randint(50000,100000,size=10),
    "city":['A','B','C','A','A','C','B','C','B','B'],
    "join_dates":pd.date_range("2020-02-01",periods=10)
})
df3.loc[[1,2,5],"age"]=np.nan
df3.loc[[2,7],"salary"]=np.nan
df3.loc[[3,5],"city"]=np.nan
df_backup=df3.copy()


In [50]:
df3['salary']=df3['salary'].ffill()
df3

,age,salary,city,join_dates
0,20.0,95663.0,A,2020-02-01
1,NaN,53912.0,B,2020-02-02
2,NaN,53912.0,C,2020-02-03
3,52.0,71752.0,NaN,2020-02-04
4,61.0,73532.0,A,2020-02-05
5,NaN,57997.0,NaN,2020-02-06
6,66.0,90800.0,B,2020-02-07
7,58.0,90800.0,C,2020-02-08
8,26.0,90899.0,B,2020-02-09
9,37.0,54845.0,B,2020-02-10


In [53]:
df4=df_backup
df4['salary']=df4['salary'].bfill()
df4

,age,salary,city,join_dates
0,20.0,95663.0,A,2020-02-01
1,NaN,53912.0,B,2020-02-02
2,NaN,71752.0,C,2020-02-03
3,52.0,71752.0,NaN,2020-02-04
4,61.0,73532.0,A,2020-02-05
5,NaN,57997.0,NaN,2020-02-06
6,66.0,90800.0,B,2020-02-07
7,58.0,90899.0,C,2020-02-08
8,26.0,90899.0,B,2020-02-09
9,37.0,54845.0,B,2020-02-10


In [ ]:
df = pd.DataFrame({
    'MedInc': [8.3, 8.0, np.nan, 5.5, 3.8, np.nan, 7.0],      
    'HouseAge': [41, 21, 52, np.nan, 52, 15, 30],              
    'MedHouseVal': [4.5, 3.5, 3.5, 3.4, np.nan, 2.8, 3.0]     
})


,MedInc,HouseAge,MedHouseVal
0,8.3,41.0,4.5
1,8.0,21.0,3.5
2,NaN,52.0,3.5
3,5.5,NaN,3.4
4,3.8,52.0,NaN
5,NaN,15.0,2.8
6,7.0,30.0,3.0


In [56]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imp=IterativeImputer()

df_imputed=pd.DataFrame(imp.fit_transform(df),columns=df.columns)

df_imputed

c:\Users\Devanshu\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,MedInc,HouseAge,MedHouseVal
0,8.300000,41.000000,4.500000
1,8.000000,21.000000,3.500000
2,5.459135,52.000000,3.500000
3,5.500000,48.897298,3.400000
4,3.800000,52.000000,2.592113
5,7.679367,15.000000,2.800000
6,7.000000,30.000000,3.000000
